This code will tokenize Pile (https://huggingface.co/datasets/monology/pile-uncopyrighted) and Lymsys (https://huggingface.co/datasets/lmsys/lmsys-chat-1m) datasets for Qwen - (For ~400 mil tokens) and upload them to HF for training 

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
from tqdm import tqdm  # Progress bar

# Load datasets with streaming enabled
pile_streamed = load_dataset("monology/pile-uncopyrighted", split="train", streaming=True)
lmsys_streamed = load_dataset("science-of-finetuning/lmsys-chat-1m-chat-formatted", split="train", streaming=True)

# Process pile: rename and add source 
def process_pile(example):
    return {"text": example["text"], "dataset": "pile"}

def process_lmsys(example):
    return {"text": example["text_qwen2_5"], "dataset": "lmsys"}

# Apply transforms and take first 500k examples with progress bars
pile_processed = map(process_pile, pile_streamed)
lmsys_processed = map(process_lmsys, lmsys_streamed)

# Convert to in-memory Dataset (limit to 500k examples each) with tqdm progress bar
pile_list = [x for x in tqdm(pile_processed, total=500_000, desc="Loading Pile", unit=" examples")][:500_000]
lmsys_list = [x for x in tqdm(lmsys_processed, total=500_000, desc="Loading LMSYS", unit=" examples")][:500_000]

# Convert to Hugging Face Dataset objects
pile_ds = Dataset.from_list(pile_list)
lmsys_ds = Dataset.from_list(lmsys_list)

# Combine the datasets
combined_dataset = concatenate_datasets([pile_ds, lmsys_ds])

# Upload to Hugging Face Hub
hf_repo_id = "AndrisWillow/pile-lmsys_qwen_format-mix-1m"
combined_dataset.push_to_hub(hf_repo_id)


In [ ]:
from transformers import AutoTokenizer
from sae_lens import PretokenizeRunner, PretokenizeRunnerConfig

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

# From https://huggingface.co/Qwen/Qwen2.5-0.5B/blob/main/tokenizer_config.json
BATCH_TOK = tok.eos_token_id                                  # 151643
# SEQ_START_TOK = tok.convert_tokens_to_ids("<|im_start|>")     # 151644
# SEQ_END_TOK  = tok.convert_tokens_to_ids("<|im_end|>")        # 151645

cfg = PretokenizeRunnerConfig(
    tokenizer_name="Qwen/Qwen2.5-0.5B",
    dataset_path="AndrisWillow/pile-lmsys_qwen_format-mix-1m",
    shuffle=True,
    num_proc=32,
    context_size=1024,
    begin_batch_token=BATCH_TOK,  
    begin_sequence_token=None, 
    sequence_separator_token=None,
    hf_repo_id="AndrisWillow/Pile-Lmsys-1m-tokenized-1024-Qwen2.5-WTemplTok",
)

dataset = PretokenizeRunner(cfg).run()
